In [1]:
from google.colab import files
uploaded = files.upload()
# pick starrydata_curves.csv

Saving starrydata_curves (1).csv to starrydata_curves (1).csv


In [3]:
import pandas as pd
import numpy as np
import json

curves = pd.read_csv('starrydata_curves (1).csv')
print("Shape:", curves.shape)
print("Columns:", list(curves.columns))
curves.head()

Shape: (189070, 15)
Columns: ['SID', 'DOI', 'composition', 'sample_id', 'figure_id', 'prop_x', 'prop_y', 'unit_x', 'unit_y', 'x', 'y', 'created_at', 'updated_at', 'project_names', 'comments']


,SID,DOI,composition,sample_id,figure_id,prop_x,prop_y,unit_x,unit_y,x,y,created_at,updated_at,project_names,comments
0,6,10.1021/am405410e,Pb1.00025Zn0.02Te1.02I0.0005,113,79,Temperature,Seebeck coefficient,K,V*K^(-1),"[299.8597,324.8683,349.8757,375.2454,399.8985,...","[-0.0001484452,-0.0001602763,-0.0001729511,-0....",Fri Sep 01 2017 18:19:39 GMT+0900 (JST),Fri Sep 01 2017 18:19:39 GMT+0900 (JST),"[""ThermoelectricMaterials""]",NaN
1,6006,10.1002/adma.201004200,Na0.0035Pb0.9965Se1,147,80,Temperature,Seebeck coefficient,K,V*K^(-1),"[317.7914,379.1411,384.0491,431.9018,452.7607,...","[0.0000727595,0.00007374922,0.00009257471,0.00...",Fri Sep 01 2017 18:19:39 GMT+0900 (JST),Fri Sep 01 2017 18:19:39 GMT+0900 (JST),"[""ThermoelectricMaterials"",""GeneralDB""]",NaN
2,6444,10.1002/anie.200803934,Pb1.03Sb0.03Te1,434,81,Temperature,Seebeck coefficient,K,V*K^(-1),"[333.1159,340.5494,347.983,355.4166,362.8501,3...","[-0.0000235374,-0.00002455838,-0.00002681692,-...",Fri Sep 01 2017 18:19:39 GMT+0900 (JST),Fri Sep 01 2017 18:19:39 GMT+0900 (JST),"[""ThermoelectricMaterials""]",NaN
3,5867,10.1016/j.actamat.2016.03.059,Pb0.76Mn0.24Te1Na0.04,248,83,Temperature,Seebeck coefficient,K,V*K^(-1),"[343.2468,376.3636,423.8961,472.2078,520.5195,...","[0.0001875,0.0002044481,0.000233961,0.00025762...",Fri Sep 01 2017 18:19:39 GMT+0900 (JST),Fri Sep 01 2017 18:19:39 GMT+0900 (JST),"[""ThermoelectricMaterials""]",NaN
4,3485,10.1021/ja109138p,Pb1S0.84Te0.16Cl0.00134,389,84,Temperature,Seebeck coefficient,K,V*K^(-1),"[303.8348,320.9667,343.9541,368.6094,393.2655,...","[-0.00009128622,-0.00009944682,-0.0001051682,-...",Fri Sep 01 2017 18:19:39 GMT+0900 (JST),Fri Sep 01 2017 18:19:39 GMT+0900 (JST),"[""ThermoelectricMaterials""]",NaN


In [4]:
curves = curves[((curves.prop_x=='Temperature')|(curves.prop_x=='T'))&(curves.prop_y=='ZT')].reset_index(drop=True)
assert len(curves[['DOI', 'sample_id', 'figure_id']].drop_duplicates()) == len(curves)
curves['curve_id'] = curves.apply(lambda curve: f'{curve["DOI"]}[{curve["sample_id"]}, {curve["figure_id"]}]', axis=1)

print("ZT-vs-T curves:", len(curves))
print("Unique DOIs:", curves['DOI'].nunique())

ZT-vs-T curves: 19352
Unique DOIs: 3855


In [5]:
PURE_SITES                 = PS = ['A', 'B', 'C']
DOPING_SITES               = DS = ['As', 'Bs', 'Cs']
SITES                      = S  = ['A', 'As', 'B', 'Bs', 'C', 'Cs']
PURE_CONCENTRATIONS        = PC = ['CA', 'CB', 'CC']
DOPING_CONCENTRATIONS      = DC = ['CAs', 'CBs', 'CCs']
CONCENTRATIONS             = C  = ['CA', 'CAs', 'CB', 'CBs', 'CC', 'CCs']
COMPOSITION                = CO = SITES + CONCENTRATIONS

def extract_hh(name):
    name = name.replace('(', '').replace(')', '').replace(' ', '').replace('-', '')
    isnumeric = lambda l: all(map(lambda c: c in '0.123456789', l))
    elements = list()
    pure_concentrations = list()
    concentrations = list()
    chunks = list()
    buffer = ''
    for i, (letter, next_letter) in enumerate(zip(name, name[1:]+' ')):
        buffer += letter
        if isnumeric(letter) != isnumeric(next_letter) or (not isnumeric(letter) and letter.upper() == letter and next_letter.upper() == next_letter) or (not isnumeric(letter) and letter.lower() == letter and next_letter.upper() == next_letter) or i == len(name):
            chunks += [buffer]
            buffer = ''
    i = len(chunks) - 1
    while i >= 0:
        if isnumeric(chunks[i]):
            concentrations += [float(chunks[i])]
            pure_concentrations += [float(chunks[i-2])]
            assert abs(1-(float(chunks[i])+float(chunks[i-2])))<1e-4
            elements += [chunks[i-1], chunks[i-3]]
            i += -4
        else:
            elements += list(reversed([chunks[i], chunks[i]]))
            pure_concentrations += [1]
            concentrations += [0]
            i += -1
    assert len(elements) == 6
    assert len(pure_concentrations) == 3
    assert len(concentrations) == 3
    elements.reverse()
    pure_concentrations.reverse()
    concentrations.reverse()
    for i in range(3):
        assert (elements[2*i]==elements[2*i+1])==(concentrations[i]==0), (i, elements, concentrations)
    result = dict(
        A=elements[0], As=elements[1], B=elements[2], Bs=elements[3], C=elements[4], Cs=elements[5],
        CA=pure_concentrations[0], CB=pure_concentrations[1], CC=pure_concentrations[2],
        CAs=concentrations[0], CBs=concentrations[1], CCs=concentrations[2],
        chunks=chunks,
    )
    return result

def check_composition(row):
    for s in PURE_SITES:
        x = row[s]
        xs = row[s+'s']
        cx = row['C'+s]
        cxs = row['C'+s+'s']
        if x == xs:
            assert cxs == 0
        else:
            assert cxs > 0
        assert cxs <= 1
        assert cx <= 1

In [6]:
print(extract_hh('TiNiSn'))

{'A': 'Ti', 'As': 'Ti', 'B': 'Ni', 'Bs': 'Ni', 'C': 'Sn', 'Cs': 'Sn', 'CA': 1, 'CB': 1, 'CC': 1, 'CAs': 0, 'CBs': 0, 'CCs': 0, 'chunks': ['Ti', 'Ni', 'Sn']}


In [7]:
rows = []
for _, row in curves.iterrows():
    comp = row['composition']
    if not isinstance(comp, str):
        continue
    try:
        parsed = extract_hh(comp)
        check_composition({**row.to_dict(), **parsed})
    except Exception:
        continue
    full_row = row.to_dict()
    full_row.update(parsed)
    rows.append(full_row)

hh_curves = pd.DataFrame(rows)
print("Curves parsed as half-Heusler-shaped composition:", len(hh_curves))

Curves parsed as half-Heusler-shaped composition: 793


In [8]:
data = list()
for _, row in hh_curves.iterrows():
    x = json.loads(row['x'])
    y = json.loads(row['y'])
    if len(x) != len(y):
        continue
    if row['prop_x'] not in ('T', 'Temperature') or row['unit_x'] != 'K':
        continue
    if row['prop_y'] != 'ZT' or row['unit_y'] != '-':
        continue
    for xv, yv in zip(x, y):
        data.append(row.to_dict() | dict(T=xv, ZT=yv))

dataset = pd.DataFrame(data)
dataset = dataset[['composition', *COMPOSITION, 'T', 'ZT', 'curve_id', 'DOI', 'sample_id', 'figure_id']].drop_duplicates(keep='first').sort_values(PURE_SITES+DOPING_SITES+CONCENTRATIONS+['DOI', 'sample_id', 'figure_id', 'T', 'ZT']).reset_index(drop=True)

print("Total (T, ZT) points:", len(dataset))

Total (T, ZT) points: 10142


In [9]:
pure_compositions = {
    str().join(elements): composition_table[
        (composition_table.A==composition_table.As) &
        (composition_table.B==composition_table.Bs) &
        (composition_table.C==composition_table.Cs)
    ]
    for elements, composition_table in list(dataset.groupby(PS))
}
pure_compositions = {name: table for name, table in pure_compositions.items() if len(table)}

for sysname in ['TiNiSn', 'NbFeSb', 'TiCoSb']:
    table = pure_compositions[sysname]
    print(f"{sysname}: {len(table)} (T,ZT) points, {table['curve_id'].nunique()} curves, {table['DOI'].nunique()} DOIs")

TiNiSn: 618 (T,ZT) points, 27 curves, 17 DOIs
NbFeSb: 86 (T,ZT) points, 10 curves, 2 DOIs
TiCoSb: 454 (T,ZT) points, 24 curves, 14 DOIs


In [10]:
def get_reference_zt_max_bin(composition_name, bin_relative_width):
    composition_table = pure_compositions[composition_name]
    zt_maximums = composition_table.groupby('curve_id')['ZT'].agg('max').reset_index()['ZT'].values
    mean_maximum = np.mean(zt_maximums)
    max_maximum = np.max(zt_maximums)
    min_maximum = np.min(zt_maximums)
    bin_width = mean_maximum * bin_relative_width
    lower_bound = mean_maximum - bin_width/2
    upper_bound = mean_maximum + bin_width/2
    bounds = [lower_bound, upper_bound]
    while lower_bound > min_maximum:
        lower_bound -= bin_width
        bounds.insert(0, lower_bound)
    while upper_bound <= max_maximum:
        upper_bound += bin_width
        bounds.append(upper_bound)

    bins = list()
    for (doi, curve_id), curve_table in composition_table.groupby(['DOI', 'curve_id']):
        zt_max = curve_table['ZT'].max()
        for low, high in zip(bounds[:-1], bounds[1:]):
            if low <= zt_max < high:
                bins.append({
                    'DOI': doi, 'curve_id': curve_id, 'bin': (low, high),
                    'low': low, 'mid': (low+high)/2, 'high': high,
                    'ZTmax': zt_max, 'Tmax': curve_table['T'].max(),
                })
    bins_content = pd.DataFrame(bins)
    bins_grouped = bins_content.groupby(['low', 'mid', 'high'])[['DOI', 'curve_id']].nunique().sort_values(['DOI', 'curve_id']).reset_index()
    reference_bin = bins_content[(bins_content['low']==bins_grouped.iloc[-1]['low'])&(bins_content['high']==bins_grouped.iloc[-1]['high'])]
    return reference_bin

In [11]:
ref_bin = get_reference_zt_max_bin('TiNiSn', 0.3)
print("Selected bin - unique DOIs:", ref_bin['DOI'].nunique())
print("Bin range:", round(ref_bin.iloc[0]['low'], 3), "-", round(ref_bin.iloc[0]['high'], 3))
print()
print(ref_bin[['DOI', 'curve_id', 'ZTmax']].to_string(index=False))

Selected bin - unique DOIs: 11
Bin range: 0.313 - 0.424

                                     DOI                                               curve_id    ZTmax
          10.1016/j.intermet.2006.08.008            10.1016/j.intermet.2006.08.008[10816, 8944] 0.366163
          10.1016/j.intermet.2006.08.008            10.1016/j.intermet.2006.08.008[10816, 8946] 0.376000
           10.1016/j.jallcom.2008.02.041             10.1016/j.jallcom.2008.02.041[11278, 9280] 0.395293
        10.1016/j.scriptamat.2020.09.010         10.1016/j.scriptamat.2020.09.010[38664, 29999] 0.401819
10.1016/j.solidstatesciences.2013.09.005 10.1016/j.solidstatesciences.2013.09.005[20678, 17242] 0.364557
10.1016/j.solidstatesciences.2013.09.005 10.1016/j.solidstatesciences.2013.09.005[20679, 17242] 0.387342
              10.1016/j.stam.2004.02.006               10.1016/j.stam.2004.02.006[33758, 26891] 0.367617
                      10.1039/c3cp50918d                        10.1039/c3cp50918d[11967, 9925] 0.37776

In [12]:
def write_material(row):
    if type(row) == tuple:
        row = dict(zip(SITES, row))
    result = ''
    for x, xs in zip(PURE_SITES, DOPING_SITES):
        if row[x] == row[xs]:
            result += row[x]
        else:
            result += f'({row[x]}{row[xs]})'
    return result

def write_composition(row):
    if type(row) == tuple:
        row = dict(zip(SITES+CONCENTRATIONS, row))
    result = ''
    for x, xs, cx, cxs in zip(PURE_SITES, DOPING_SITES, PURE_CONCENTRATIONS, DOPING_CONCENTRATIONS):
        if row[cxs] == 0:
            result += row[x]
        else:
            result += f'{row[x]}<sub>{row[cx]}</sub>{row[xs]}<sub>{row[cxs]}</sub>'
    return result

In [14]:
def get_reference_doi(composition_name, reference_bin):
    pure_sites_elements = tuple(pure_compositions[composition_name].iloc[0][PURE_SITES])
    information_gain_data = list()
    for doi in reference_bin.DOI.unique():
        extended_dataset = dataset[dataset.apply(
            lambda row: row['DOI']==doi and tuple(row[PURE_SITES]) == pure_sites_elements and tuple(row[DOPING_SITES]) != pure_sites_elements,
            axis=1,
        )]
        information_gain_data.append(dict(DOI=doi, material=None, composition=None))
        for _, row in extended_dataset[COMPOSITION].drop_duplicates(keep='first').iterrows():
            information_gain_data.append(dict(DOI=doi, material=write_material(row), composition=write_composition(row)))
    information_gain_content = pd.DataFrame(information_gain_data, columns=['DOI', 'material', 'composition'])
    information_gain_counts = information_gain_content.groupby(['DOI'])[['material', 'composition']].nunique().sort_values(['material', 'composition']).reset_index()
    top_information_gain_counts = information_gain_counts[
        (information_gain_counts['material']==information_gain_counts.iloc[-1]['material']) &
        (information_gain_counts['composition']==information_gain_counts.iloc[-1]['composition'])
    ]
    if len(top_information_gain_counts) == 1:
        reference_doi = top_information_gain_counts.DOI.unique()
    else:
        temperatures = reference_bin.groupby(['DOI'])['Tmax'].agg('max').reset_index().sort_values('Tmax')
        top_temperatures = temperatures[temperatures['Tmax']==temperatures.iloc[-1]['Tmax']]
        reference_doi = [top_temperatures.iloc[-1]['DOI']]
    return reference_doi

def get_reference_curve(composition_name, reference_bin, reference_doi):
    target_max_zt = reference_bin['ZTmax'].mean()
    doi_curves = reference_bin[reference_bin['DOI'].apply(lambda doi: doi in reference_doi)].copy()
    doi_curves['Distance'] = abs(doi_curves['ZTmax']-target_max_zt)
    doi_curves = doi_curves.sort_values(['Distance', 'ZTmax'], ascending=[True, False])
    top_doi_curves = doi_curves[
        (doi_curves['Distance']==doi_curves.iloc[0]['Distance']) & (doi_curves['ZTmax']==doi_curves.iloc[0]['ZTmax'])
    ]
    return top_doi_curves.iloc[0]

In [15]:
ref_doi = get_reference_doi('TiNiSn', ref_bin)
print("Reference DOI:", ref_doi[0])

ref_curve = get_reference_curve('TiNiSn', ref_bin, ref_doi)
print("Reference curve:", ref_curve['curve_id'])
print("zT_max_ref =", round(ref_curve['ZTmax'], 4))

Reference DOI: 10.1016/j.intermet.2006.08.008
Reference curve: 10.1016/j.intermet.2006.08.008[10816, 8946]
zT_max_ref = 0.376


In [16]:
for system in ['NbFeSb', 'TiCoSb']:
    print(f"=== {system} ===")
    table = pure_compositions[system]
    zt_max_per_curve = table.groupby(['DOI','curve_id'])['ZT'].agg('max').reset_index()
    print(zt_max_per_curve.sort_values('ZT', ascending=False).to_string(index=False))
    print()

=== NbFeSb ===
                          DOI                                    curve_id       ZT
       10.1002/advs.201800278        10.1002/advs.201800278[15632, 13248] 0.671429
       10.1002/advs.201800278        10.1002/advs.201800278[15632, 13252] 0.650000
       10.1002/advs.201800278        10.1002/advs.201800278[15630, 13248] 0.592063
       10.1002/advs.201800278        10.1002/advs.201800278[15630, 13252] 0.539286
       10.1002/advs.201800278        10.1002/advs.201800278[15631, 13248] 0.519048
       10.1002/advs.201800278        10.1002/advs.201800278[15631, 13252] 0.471429
10.1016/j.actamat.2019.11.010 10.1016/j.actamat.2019.11.010[35329, 27928] 0.000430
10.1016/j.actamat.2019.11.010 10.1016/j.actamat.2019.11.010[35327, 27928] 0.000251
10.1016/j.actamat.2019.11.010 10.1016/j.actamat.2019.11.010[35326, 27928] 0.000230
10.1016/j.actamat.2019.11.010 10.1016/j.actamat.2019.11.010[35328, 27928] 0.000216

=== TiCoSb ===
                          DOI                           

In [17]:
for system in ['NbFeSb', 'TiCoSb']:
    ref_bin_x = get_reference_zt_max_bin(system, 0.3)
    print(f"{system}: selected bin has {ref_bin_x['DOI'].nunique()} unique DOIs, mean ZTmax = {round(ref_bin_x['ZTmax'].mean(), 4)}")

NbFeSb: selected bin has 1 unique DOIs, mean ZTmax = 0.0003
TiCoSb: selected bin has 8 unique DOIs, mean ZTmax = 0.0313
